In [1]:
import glob, os, json, time, random, pickle, math, re
import pandas as pd
import numpy as np
import torch

import matplotlib.pyplot as plt
import cv2, PIL, base64
from moviepy import VideoFileClip

from tqdm import tqdm, trange
from functools import partial
from pathlib import Path
# from motionBERT import DSTformer, get_config, coco2h36m, render_and_save, adjust_head
from ultralytics import YOLO

import data_utils

os.environ['KMP_DUPLICATE_LIB_OK']='True'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [2]:
# url = "https://www.youtube.com/shorts/1v8vSzpEu_A"
# https://www.youtube.com/shorts/v2nmG7THITc
url = "https://www.youtube.com/watch?v=O9_sXSNhOlA"
download_dir = "data/paper_figure"
data_utils.download_video(url, download_dir)

[youtube] Extracting URL: https://www.youtube.com/watch?v=O9_sXSNhOlA
[youtube] O9_sXSNhOlA: Downloading webpage


[youtube] O9_sXSNhOlA: Downloading android vr player API JSON
[info] O9_sXSNhOlA: Downloading 1 format(s): 313+251
[download] Destination: data\paper_figure\Homestead_Building__Installing_a_Window_in_the_Mini_House.f313.webm
[download] 100% of  411.31MiB in 00:00:25 at 16.14MiB/s    
[download] Destination: data\paper_figure\Homestead_Building__Installing_a_Window_in_the_Mini_House.f251.webm
[download] 100% of    2.68MiB in 00:00:00 at 2.85MiB/s   
[Merger] Merging formats into "data\paper_figure\Homestead_Building__Installing_a_Window_in_the_Mini_House.mp4"
Deleting original file data\paper_figure\Homestead_Building__Installing_a_Window_in_the_Mini_House.f313.webm (pass -k to keep)
Deleting original file data\paper_figure\Homestead_Building__Installing_a_Window_in_the_Mini_House.f251.webm (pass -k to keep)


'data/paper_figure\\Homestead_Building__Installing_a_Window_in_the_Mini_House.mp4'

In [3]:
# source_dir = r"C:\Users\17346\src\ergolmm\data\videos"
# source_dir = r"C:\Users\17346\src\VEHS\YoutubeData_post"
# source_dir = r"C:\Users\17346\src\VEHS\YoutubeData_newPost"
# source_dir = r"C:\Users\17346\src\ConstActLoc\data\youtube"
# target_dir = r"C:\Users\17346\src\ConstActLoc\data\clipped"
source_dir = "data/paper_figure"
target_dir = "data/paper_figure"

source_video_filename = "Homestead_Building__Installing_a_Window_in_the_Mini_House" + ".mp4"
source_video_path = os.path.join(source_dir, source_video_filename)

start_time = 68
end_time = 159
target_video_path = os.path.join(target_dir, f"clipped_{start_time}_{end_time}_{source_video_filename}")

data_utils.clip_video(source_video_path, start_time, end_time, target_video_path)

{'video_found': True, 'audio_found': True, 'metadata': {'major_brand': 'isom', 'minor_version': '512', 'compatible_brands': 'isomiso2mp41', 'encoder': 'Lavf58.45.100'}, 'inputs': [{'streams': [{'input_number': 0, 'stream_number': 0, 'stream_type': 'video', 'language': 'eng', 'default': True, 'size': [3840, 1920], 'bitrate': 14844, 'fps': 29.97002997002997, 'codec_name': 'vp9', 'profile': '(Profile 0)', 'metadata': {'Metadata': '', 'handler_name': 'VideoHandler', 'vendor_id': '[0][0][0][0]'}}, {'input_number': 0, 'stream_number': 1, 'stream_type': 'audio', 'language': 'eng', 'default': True, 'fps': 48000, 'bitrate': 94, 'metadata': {'Metadata': '', 'handler_name': 'SoundHandler', 'vendor_id': '[0][0][0][0]'}}], 'input_number': 0}], 'duration': 232.42000000000002, 'bitrate': 14944, 'start': 0.0, 'default_video_input_number': 0, 'default_video_stream_number': 0, 'video_codec_name': 'vp9', 'video_profile': '(Profile 0)', 'video_size': [3840, 1920], 'video_bitrate': 14844, 'video_fps': 29.9

MoviePy - Done.
MoviePy - Writing video data/paper_figure\clipped_68_159_Homestead_Building__Installing_a_Window_in_the_Mini_House.mp4



MoviePy - Done !
MoviePy - video ready data/paper_figure\clipped_68_159_Homestead_Building__Installing_a_Window_in_the_Mini_House.mp4


In [3]:
source_dir = Path(r"C:\Users\17346\src\ConstActLoc\data\clipped")
target_dir_fps1 = Path(r"C:\Users\17346\src\ConstActLoc\data\frames_fps1")
target_dir_fps8 = Path(r"C:\Users\17346\src\ConstActLoc\data\frames_fps8")
existing_files = [f.name for f in target_dir_fps1.glob("*.jpg")]

mp4_files = sorted(source_dir.glob("*.mp4"))
print(f"Found {len(mp4_files)} mp4 files.")

num_skipped = 0
for video_path in tqdm(mp4_files, desc="Processing videos"):
    video_name = re.escape(video_path.stem)
    pattern = re.compile(rf"^{video_name}_\d+_\d+\.jpg$")
    already_done = any(pattern.match(fname) for fname in existing_files)

    if already_done:
        num_skipped += 1
        continue

    data_utils.extract_frames(video_path, target_fps=8)

print(f"All done. Skipped {num_skipped} videos.")

Found 100 mp4 files.


Processing videos: 100%|██████████| 100/100 [06:22<00:00,  3.83s/it]

All done. Skipped 91 videos.


In [5]:
video_path = "data/paper_figure/clipped_68_159_Homestead_Building__Installing_a_Window_in_the_Mini_House.mp4"
# video_name = re.escape(Path(video_path).stem)
cap = cv2.VideoCapture(str(video_path))

original_fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration = total_frames / original_fps if original_fps > 0 else 0

video_name = Path(video_path).stem
target_fps = 1
target_dir_fps1 = Path("data/paper_figure/frames")

for sec in range(math.floor(duration)):

    success_count = 0

    for i in range(target_fps):

        t = sec + (i + 0.5) / target_fps
        frame_idx = int(round(t * original_fps))

        if frame_idx >= total_frames:
            continue

        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()

        if not ret:
            print(
                f"[FAILED FRAME] "
                f"video={video_name}, sec={sec}, "
                f"frame_idx={frame_idx}"
            )
            continue

        success_count += 1

        filename = f"{video_name}_{sec}_{frame_idx}.jpg"

        # 1 fps 저장
        cv2.imwrite(str(target_dir_fps1 / filename), frame)

In [ ]:
import json
API_KEY_JSON_PATH = "APIKEY/api_key.json"
API_KEY_JSON = json.load(open(API_KEY_JSON_PATH, "r"))
OPENAI_API_KEY = API_KEY_JSON["OpenAI_yong"]
CLAUDE_API_KEY = API_KEY_JSON["Anthropic_yong"]
GEMINI_API_KEY = API_KEY_JSON["Gemini_yong"]

from genais import AgentOpenAI, AgentClaude, AgentGemini
# frame_path = "data/frames_fps1/clipped_0_11_cart_v1_8_214.jpg"
# frame_path = "data/demo.jpg"
# agent = AgentOpenAI(model_name="gpt-5.4", api_key=OPENAI_API_KEY)
# agent = AgentClaude(model_name="claude-haiku-4-5-20251001", api_key=CLAUDE_API_KEY)
agent = AgentGemini(model_name="gemini-3.1-flash-lite-preview", api_key=GEMINI_API_KEY)

# action, inference_cost, inference_time = agent.inference_one_frame(frame_path)
# print(action, inference_cost, inference_time)

video_path = "data/paper_figure/clipped_68_159_Homestead_Building__Installing_a_Window_in_the_Mini_House.mp4"
segments, inference_cost, inference_time = agent.inference_one_video(video_path)
segments = [(seg.action, seg.start_second) for seg in segments]
print(segments, inference_cost, inference_time)
# Gemini Pro: [('none', 0), ('lift/carry window/sheets', 33), ('climb/climb down ladders/steps', 36), ('none', 41), ('hammer', 123), ('none', 127)] 0.018824 58.83242082595825
# Gemini Pro: [('none', 68), ('lift/carry window/sheets', 101), ('climb/climb down ladders/steps', 104), ('none', 109), ('hammer', 121), ('none', 127)] 0.018824 58.83242082595825
# Gemini Flash Lite: [('none', 0), ('apply adhesive/mortar', 16), ('none', 19), ('lift/carry window/sheets', 22), ('none', 24), ('lift/carry window/sheets', 32), ('none', 50), ('hammer', 123), ('none', 126)] 0.002461 46.3207483291626
# Gemini Flash Lite: [('none', 68), ('apply adhesive/mortar', 84), ('none', 87), ('lift/carry window/sheets', 90), ('none', 92), ('lift/carry window/sheets', 100), ('none', 118), ('hammer', 191), ('none', 193)] 0.002461 46.3207483291626

[('none', 0), ('apply adhesive/mortar', 16), ('none', 19), ('lift/carry window/sheets', 22), ('none', 24), ('lift/carry window/sheets', 32), ('none', 50), ('hammer', 123), ('none', 126)] 0.002461 46.3207483291626
